In [10]:
import os
import cv2

data_path = "../data"

images = []
labels = []

for label in os.listdir(data_path):
    class_path = os.path.join(data_path, label)
    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)
            img = cv2.imread(img_path)
            images.append(img)
            labels.append(label)

print("Total images:", len(images))
print("Labels:", set(labels))

Total images: 15
Labels: {'CA1', 'CA3', 'Dentate_Gyrus'}


In [11]:
import os
import cv2

data_path = "../data"

images = []
labels = []

for label in os.listdir(data_path):
    class_path = os.path.join(data_path, label)
    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)
            img = cv2.imread(img_path)
            images.append(img)
            labels.append(label)

print("Total images:", len(images))
print("Labels:", set(labels))

Total images: 15
Labels: {'CA1', 'CA3', 'Dentate_Gyrus'}


In [15]:
import numpy as np

In [17]:
import cv2

processed_images = []

for img in images:
    img_resized = cv2.resize(img, (128,128))
    processed_images.append(img_resized)

print("Processed images:", len(processed_images))
print("Example shape:", processed_images[0].shape)

Processed images: 15
Example shape: (128, 128, 3)


In [18]:
import numpy as np

X = np.array(processed_images)
X = X.reshape(len(X), -1)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (15, 49152)


In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(labels)

print("Encoded labels:", y)
print("Classes:", le.classes_)

Encoded labels: [1 1 1 0 0 0 0 0 0 0 0 2 2 2 2]
Classes: ['CA1' 'CA3' 'Dentate_Gyrus']


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# create model
model = RandomForestClassifier()

# train model
model.fit(X_train, y_train)

# evaluate accuracy
accuracy = model.score(X_test, y_test)

print("Model accuracy:", accuracy)

Model accuracy: 1.0


In [21]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("True labels:", y_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

Predictions: [0 2 1 2 0]
True labels: [0 2 1 2 0]
Confusion Matrix:
 [[2 0 0]
 [0 1 0]
 [0 0 2]]


In [22]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=le.classes_))

               precision    recall  f1-score   support

          CA1       1.00      1.00      1.00         2
          CA3       1.00      1.00      1.00         1
Dentate_Gyrus       1.00      1.00      1.00         2

     accuracy                           1.00         5
    macro avg       1.00      1.00      1.00         5
 weighted avg       1.00      1.00      1.00         5



In [23]:
import skimage

In [24]:
conda install scikit-image

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [25]:
from skimage.feature import graycomatrix, graycoprops
import numpy as np
import cv2

texture_features = []

for img in processed_images:

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    glcm = graycomatrix(gray,
                        distances=[1],
                        angles=[0],
                        levels=256,
                        symmetric=True,
                        normed=True)

    contrast = graycoprops(glcm, 'contrast')[0,0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0,0]
    energy = graycoprops(glcm, 'energy')[0,0]
    correlation = graycoprops(glcm, 'correlation')[0,0]

    texture_features.append([contrast, homogeneity, energy, correlation])

X_texture = np.array(texture_features)

print("Texture feature shape:", X_texture.shape)

Texture feature shape: (15, 4)


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_texture, y, test_size=0.3, random_state=42
)

model = RandomForestClassifier()

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("Texture feature accuracy:", accuracy)

Texture feature accuracy: 1.0
